In [1]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping
import keras_tuner as kt
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score

In [2]:
df=pd.read_csv('Food_Delivery_Time_Prediction.csv')

In [3]:
df.shape

(50000, 24)

In [4]:
df=df.drop(columns=['Order_ID', 'Order_Date'])

In [5]:
df.select_dtypes(exclude='object').columns

Index(['Order_Hour', 'Is_Weekend', 'Is_Festival', 'Rider_Experience_Years',
       'Rider_Rating', 'Restaurant_Rating', 'Order_Items',
       'Preparation_Time_Min', 'Road_Distance_km', 'Number_of_Signals',
       'Average_Speed_kmph', 'Time_taken_min'],
      dtype='object')

In [6]:
def get_features():
    nom_cols=['Day_of_Week', 'Weather', 'Pickup_Zone', 'Dropoff_Zone', 'Vehicle_Type', 'Cuisine_Type']
    ord_cols=['Restaurant_Load', 'Delivery_Distance_Category', 'Traffic_Level', 'Delivery_Priority']
    num_cols=['Order_Hour', 'Is_Weekend', 'Is_Festival', 'Rider_Experience_Years', 'Rider_Rating',
            'Restaurant_Rating', 'Order_Items', 'Preparation_Time_Min', 'Road_Distance_km',
            'Number_of_Signals', 'Average_Speed_kmph']
    return num_cols, ord_cols, nom_cols

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

def get_preprocessor(numerical_data, ordinal_data, nominal_data):
    num_pipeline = Pipeline(
        steps=[
            ('impute', SimpleImputer(strategy='mean')),
            ('scaling', StandardScaler())
        ]
    )

    ord_pipeline = Pipeline(
        steps=[
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('encoding', OrdinalEncoder(
                categories=[
                    ['Low', 'Medium', 'High'],
                    ['Short', 'Medium', 'Long'],
                    ['Low', 'Moderate', 'High', 'Severe'],
                    ['Normal', 'Priority', 'VIP']                
                ],
                handle_unknown='use_encoded_value', 
                unknown_value=-1
            )),
            ('scaling', StandardScaler())
        ]
    )

    nom_pipeline = Pipeline(
        steps=[
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('encoding', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]
    )

    features = ColumnTransformer(
        transformers=[
            ('Numerical Pipeline', num_pipeline, numerical_data),
            ('Ordinal Pipeline', ord_pipeline, ordinal_data),
            ('Nominal Pipeline', nom_pipeline, nominal_data)
        ]
    )

    return features

In [8]:
X=df.iloc[:,:-1]
y=df['Time_taken_min']

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

In [10]:
num_data, ord_data, nom_data = get_features()
features = get_preprocessor(num_data, ord_data, nom_data)

In [11]:
X_train=features.fit_transform(X_train)
X_test=features.transform(X_test)
X_val=features.transform(X_val)

In [12]:
def build_model(hp):
    model = Sequential()
    model.add(Input(shape=(X_train.shape[1],)))
    num_layers=hp.Int('hidden', min_value=1, max_value=10, step=1)

    for i in range(num_layers):
        nodes=hp.Int(f'nodes_{i}', min_value=32, max_value=512, step=32)
        model.add(Dense(nodes, activation='relu'))

        dropout_val=hp.Float(f'dropout_{i}', min_value=0.0, max_value=0.8, step=0.1)
        if dropout_val>0.0:
            model.add(Dropout(dropout_val))

    model.add(Dense(1))

    opt=hp.Choice('optimizer', values=['RMSprop', 'Adam', 'AdamW', 'Adagrad'])

    model.compile(optimizer=opt, loss='mean_squared_error', metrics=['mean_absolute_error'])

    return model

In [13]:
tuner=kt.RandomSearch(
    build_model,
    objective='val_loss',
    max_trials=20,
    directory='Food_Delivery_Time_Prediction',
    project_name='Time_Taken_Prediction',
    overwrite=True
)

In [14]:
tuner.search(X_train, y_train, epochs=10, validation_data=(X_val, y_val))

Trial 20 Complete [00h 01m 34s]
val_loss: 690.299072265625

Best val_loss So Far: 19.13442039489746
Total elapsed time: 00h 31m 42s


In [15]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
best_model=tuner.hypermodel.build(best_hp)

In [16]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

print("Training final Neural Network...")
history = best_model.fit(
    X_train, y_train,
    epochs=50, 
    validation_data=(X_val, y_val),
    callbacks=[early_stop],
    verbose=1,
)

Training final Neural Network...
Epoch 1/50


1000/1000 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - loss: 420.1620 - mean_absolute_error: 10.8656 - val_loss: 35.5068 - val_mean_absolute_error: 3.9951
Epoch 2/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - loss: 73.7892 - mean_absolute_error: 6.2522 - val_loss: 35.0832 - val_mean_absolute_error: 4.0385
Epoch 3/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - loss: 68.1543 - mean_absolute_error: 5.9629 - val_loss: 28.7043 - val_mean_absolute_error: 3.6037
Epoch 4/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - loss: 62.6857 - mean_absolute_error: 5.6774 - val_loss: 26.5559 - val_mean_absolute_error: 3.3794
Epoch 5/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - loss: 60.0461 - mean_absolute_error: 5.5609 - val_loss: 24.2730 - val_mean_absolute_error: 3.2529
Epoch 6/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - loss: 57.9616 - mean_absolute_error: 5.4456 - val_loss: 36.3550 - val_mean_absolute_error: 4.1726
Epoch 7/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - loss: 55.8748 - mean_absolut

In [17]:
print("Predicting on test data...")
y_pred = best_model.predict(X_test).flatten()

Predicting on test data...
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step


In [18]:
nn_test_mse = mean_squared_error(y_test, y_pred)
nn_test_r2 = r2_score(y_test, y_pred)

In [19]:
print(f"Neural Network Test MSE: {nn_test_mse:.4f}")
print(f"Neural Network Test R2 Score: {nn_test_r2:.4f}")

Neural Network Test MSE: 13.6087
Neural Network Test R2 Score: 0.9893


In [20]:
best_model.save('delivery_nn_model.keras')